[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch

In [31]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
      # temperature, top-k filter, top-p filter, sample
      logits = logits / temperature #[B, vo]

      if top_k > 0:
        top_values, top_idx = torch.topk(logits, top_k, dim=-1)
        logits = torch.full_like(logits, float("-inf"))
        logits.scatter_(-1, top_idx, top_values)

      if top_p < 1.0:

        sorted_values, sorted_idx = torch.sort(logits, -1, descending=True)
        sorted_prob = sorted_values.softmax(dim=-1)
        sorted_prob_cumsum = sorted_prob.cumsum(dim=-1)
        sorted_values = sorted_values.masked_fill(sorted_prob_cumsum-sorted_prob>=top_p, float("-inf"))
        logits = torch.zeros_like(logits)       # ✅ 加这行：清零
        logits.scatter_(-1, sorted_idx, sorted_values)

      probs = logits.softmax(dim=-1)



      idx = torch.multinomial(probs, 1)

      return idx.item()








In [32]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [33]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')



🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (3.5ms)
  ✅ [2/4] Low temperature concentrates (6.7ms)
  ✅ [3/4] All tokens reachable (no filtering) (119.6ms)
  ✅ [4/4] Returns valid index (3.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (133.6ms total)
  Progress saved. Run status() to see your dashboard.

